# Canonical Model 08: Recalibration Rehearsal

A dress rehearsal of the Ten Trails recalibration, on a model small enough to run in a
few minutes and with an answer known in advance. `build_rehearsal` builds a model with
Ten Trails' **shape** -- DISV, five layers, NEWTON, an array RCH on an average-year cycle,
two RIV packages (`riv` and `crisp`), a DRN, a GHB, a CHD, layered storage and two
excavated lakes under one LAK package -- twice, in one `mf.Project`: a **start** and a
**truth** that differs from it by known multipliers (layer-2 K by zone, wet- and
dry-season recharge, Sy, crisp's conductance, one lake's bed leakance). The truth is
declared through the package-first API, never through PEST, so a parameter routed to the
wrong package, lake, month or layer cannot cancel out.

The workflow below is written once, in the API to copy for Ten Trails. Only the data
are synthetic: here they are read off the truth run, there they are your hydrograph,
lake-stage and gauge files. Each step is a check worth keeping:

1. run the start model;
2. bin dated readings onto the model's periods (`from_hydrographs`, average-year mode);
3. check the cycle has settled (`AverageYear.equilibrium`);
4. declare the calibration -- with a FLUX target, because heads alone cannot tell K from
   recharge;
5. **the truth round trip, before any IES**: the template, run at the truth's
   multipliers, must reproduce the noise-free data;
6. history-match the noisy data with PESTPP-IES;
7. read what came back against the truth;
8. take the calibrated base realization home, run it, and check it reproduces what IES ran.

The knobs are in the first cell; a bigger ensemble is a better answer and a longer run.

In [ ]:
import os
from pathlib import Path

REALS = 16                       # PESTPP-IES realizations
ITERATIONS = 2                   # PESTPP-IES iterations
WORKERS = min(6, os.cpu_count() or 1)
CYCLES = 3                       # average-year cycles after the steady period (>= 2)
SEED = 2026                      # the data's noise and the prior draw
NOISE = {"head": 0.05, "stage": 0.02, "flow": 0.05}   # ft, ft, and a fraction of the flow
PP_SPACING = 400.0               # ft between the layer-2 K pilot points
WORKSPACE = Path("../artifacts/canonical_08_rehearsal")

In [ ]:
import shutil
import sys

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / "src"
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import numpy as np
import pandas as pd

import myflopy as mf
from canonical_notebook_style import notebook_header
from myflopy.modflow.mf6 import canonical_rehearsal as rehearsal

notebook_header("08", "Recalibration Rehearsal",
                "A Ten Trails-shaped truth, calibrated back through the API the recalibration uses.")

## 1. The case: a start and a truth in one project

The project's package library holds the start's `npf`, `sto`, `rch`, `crisp` and `lak`
and a `<key>/truth` variant of each; the truth simulation is
`project.simulations.fork("start", "truth", swap=...)`. Layer 2's K in the truth is
`mf.RasterValues("k_gravel.tif") * mf.Polygons(zone_1, value=2.0, default=1.0) *
mf.Polygons(zone_2, value=0.5, default=1.0)`.

In [ ]:
shutil.rmtree(WORKSPACE, ignore_errors=True)
case = rehearsal.build_rehearsal(WORKSPACE, cycles=CYCLES)
project = case.project
display(project.simulations.summary())
pd.Series(case.truth_multipliers, name="truth: x the start")

## 2. Run the start model

What the user has for Ten Trails: a model, run. `mf.AverageYear.from_model` reads the
cycle back from the run's own TDIS and STO.

In [ ]:
run = project.prepare_run("start", case.start_simulation, overwrite=True)
run.execute()
flow = run.model(case.model_name)
year = mf.AverageYear.from_model(flow)
year

## 3. The data

Dated readings, as a field program exports them: a wide table of well hydrographs (a
`date` column, one column per well, several readings a month over five years, some
months missing), each lake's stage readings, and the crisp creek's gauged baseflow by
month -- in MF6's sign, negative where the aquifer feeds the creek (ledger 350). Two
wells are screened across layers 1 and 2. Here the readings are read off the truth run
with noise added; for Ten Trails they are your files.

In [ ]:
heads_csv = rehearsal.synthetic_hydrographs(case, noise_sd=NOISE["head"], seed=SEED)
series = rehearsal.synthetic_series(case, stage_sd=NOISE["stage"], flow_cv=NOISE["flow"], seed=SEED)
display(case.wells.drop(columns="geometry"))
pd.read_csv(heads_csv, parse_dates=["date"]).head()

## 4. Has the cycle settled?

An average-year model compares targets on its LAST cycle, which is only meaningful once
the heads repeat from one year to the next. `year.equilibrium(model)` compares the last
two cycles, period end against period end, everywhere. Here two cycles are not enough
(the steady period is not the periodic state: the truth's first two cycles differ by
0.59 ft at a lake), three are (0.006 ft).

In [ ]:
equilibrium = year.equilibrium(case.truth_model(), tolerance=NOISE["head"])
display(equilibrium.summary())
print("settled within", NOISE["head"], "ft:", equilibrium.converged)

## 5. The calibration, declared once

The same declaration serves the round trip (on noise-free data) and the history match
(on the noisy data), so it is a function of the data.

- **K**: pilot points on layer 2, one set per zone (`zones=`), a correlated prior.
- **Recharge**: one multiplier for the wet months and one for the dry, over every cycle
  (`periods=`); the steady period is in neither group.
- **Sy**, **crisp's conductance** (`package="crisp"`: the model has two RIV packages, and
  a call naming none is refused), and **each lake's bed leakance** (`style="feature"`).
- **Targets**: the binned heads (a screened well is one T-weighted head), each
  hydrograph's shape (`differences()`), the lake stages, **crisp's baseflow** -- the flux
  that separates K from recharge -- and a one-sided bound (no lake above its rim).
  Weights are 1 / the noise.
- **Forecast**: a head nobody measured, at the end of the run.

In [ ]:
def calibration(name, heads_csv, series):
    heads = mf.HeadTargets.from_hydrographs(case.wells, heads_csv, model=flow, mode="average_year")
    lakes = mf.LakeStageTargets.from_hydrographs(
        {lake: lake for lake in case.lakes}, series["lake_stage"], model=flow, mode="average_year")
    gauged = pd.read_csv(series["crisp_flow"])
    baseflow = mf.RivFlowTargets(
        locations=[{"name": "crisp", "cells": list(case.crisp_cells),
                    "weight": 1.0 / (NOISE["flow"] * gauged["crisp"].abs().mean())}],
        values=gauged, package="crisp")
    last_cycle = sorted(year.months_of(-1).values())
    rims = mf.LakeStageTargets(
        locations={lake: lake for lake in case.lakes},
        values=pd.DataFrame({"time": last_cycle, **case.lake_rims}),
        inequality="at_most")

    cal = flow.pest(name)
    cal.parameterize("k", style="pilotpoints", layers=[2], zones=case.zone_spec(flow),
                     pp_spacing=PP_SPACING, correlation=800.0, bounds=(0.2, 5.0), name="kg")
    cal.parameterize("recharge", periods=case.month_groups(), bounds=(0.4, 2.5), name="rch")
    cal.parameterize("sto.sy", bounds=(0.3, 3.0), name="sy")
    cal.parameterize("riv.cond", package="crisp", bounds=(0.1, 10.0), name="crisp")
    cal.parameterize("lak.bedleak", bounds=(0.1, 10.0), name="bedleak")

    cal.observe(heads)
    cal.observe(heads.differences())
    cal.observe(mf.LakeStageObservationSpec(targets=lakes, weight=1.0 / NOISE["stage"]))
    cal.observe(baseflow)
    cal.observe(rims, prefix="rim")
    cal.forecast(case.forecast, prefix="fore")
    cal.build(f"{name}.pst", noptmax=0)
    return cal

## 6. The truth round trip -- before any IES

Run the template once at the truth's multipliers (`truth_parameter_values` reads which
parameter is which from the build's own record, never from file names) and compare with
the NOISE-FREE data. Every parameter that reaches the wrong package, lake, month or layer,
and every observation that reads the wrong row or well, shows here as a residual. The
start -- every multiplier at 1 -- is run the same way for scale. For Ten Trails, the same
check with a hand-set parameter set is the cheapest test of a new template.

In [ ]:
# THE TRUTH ROUND TRIP
check = calibration("truthcheck", rehearsal.synthetic_hydrographs(case), rehearsal.synthetic_series(case))
truth_values = rehearsal.truth_parameter_values(case, check)
start_values = dict(zip(check.pst.parameter_data["parnme"].str.lower(),
                        check.pst.parameter_data["parval1"].astype(float)))
truth_simulated = rehearsal.run_template(check, truth_values, WORKSPACE / "round_trip" / "truth")
truth_trip = rehearsal.round_trip(check, truth_simulated)
start_trip = rehearsal.round_trip(
    check, rehearsal.run_template(check, start_values, WORKSPACE / "round_trip" / "start"))
pd.DataFrame({"truth: max error": truth_trip.groupby("group")["error"].max(),
              "truth: max error / allowed": truth_trip.groupby("group")["ratio"].max(),
              "start: max error": start_trip.groupby("group")["error"].max()})

## 7. History-match the noisy data

In [ ]:
cal = calibration("rehearsal", heads_csv, series)
ies = cal.run_ies(reals=REALS, iterations=ITERATIONS, workers=WORKERS, seed=SEED, overwrite=True,
                  ies_lambda_mults=1.0, lambda_scale_fac=1.0)
print(ies.settings)
ies.plot_phi()

In [ ]:
ies.plot_vs_obs(max_locations=6)

## 8. Did the truth come back?

Each scalar parameter's truth against its prior and posterior 5-95% bands, and how much
of the log distance from the start (1.0) to the truth the posterior mean covered, and `z`,
the truth's distance from the posterior mean in posterior standard deviations (log10).
The pilot-point K field is not judged cell by cell: at this budget the data cannot pin it
(ledger 589), so it is judged by what it predicts -- the forecast.

**Read the bands with care.** At this budget the posterior is not a converged answer.
Phi falls to 1-2% of the prior's, yet the realization IES ends on fits the noisy data
1.6-3 times worse than the truth's multipliers do (phi 640-1050 against 350-390 at 16
realizations x 2 iterations, measured), and the ensemble has collapsed around where it
stopped: five or six of the six scalar truths sit outside their 5-95% bands, some at
|z| of 5-18 (ledger 590). At this budget it is not the `differences()`:
halving both head weights left the same truths outside. For Ten Trails, a narrow band
after a few iterations is not a measured uncertainty -- score the template at a hand-set
parameter set (the round trip above) against the posterior's phi before believing it.

**Converged, the truth comes back.** With PESTPP-IES's own lambda testing (drop
`ies_lambda_mults` and `lambda_scale_fac` from the `run_ies` call) and 64 realizations x
3-6 iterations, the base fits the noisy data better than the truth does, the forecast's
truth is inside its posterior range on every seed measured and 28 of 36 scalar truths (six
seeds) sit inside their bands; most misses are the linear posterior's own (the noise
drawn), but the dry-month recharge multiplier's spread is 0.7-0.9x the linear posterior's,
so its band is somewhat optimistic (ledger 591). There the `differences()` do show: beside the levels they count
each reading's noise twice in the hydrograph's shape, and the seasonal recharge and Sy
bands come back 1.15-1.33x too narrow (ledger 592).

In [ ]:
recovered = rehearsal.recovery(case, cal, ies)
forecast = ies.forecast(0)
forecast_truth = float(truth_simulated[forecast.name])
display(recovered)
print(f"forecast {forecast.name}: truth {forecast_truth:.3f}; posterior "
      f"{forecast.posterior.min():.3f} .. {forecast.posterior.max():.3f}; prior "
      f"{forecast.prior.min():.3f} .. {forecast.prior.max():.3f}")

## 9. Take the calibrated model home, and check it

`ies.calibrated()` replays the base realization's forward run and reads back what its
MODFLOW 6 read; `to_library` registers those as new library packages and forks the start
simulation onto them. The rebuilt model is run, and its outputs are read by the
calibration's own readers (`run_template(..., outputs_from=rebuilt)`) -- they must match
what PESTPP-IES ran for the base realization.

In [ ]:
calibrated = ies.calibrated()
display(calibrated.summary())
keys = calibrated.to_library(project, simulation="calibrated", overwrite=True)
rebuilt = project.prepare_run("calibrated", "calibrated", overwrite=True)
rebuilt.execute()
rebuilt_simulated = rehearsal.run_template(cal, calibrated, WORKSPACE / "round_trip" / "write_back",
                                           outputs_from=rebuilt)
write_back = rehearsal.round_trip(cal, rebuilt_simulated, rehearsal.base_observations(ies))
write_back.groupby("group")[["error", "ratio"]].max()

## 10. The rehearsal report

The pass criteria (the user's, Q9): the two round trips (each observation's error
over what is allowed it, `rehearsal.ROUND_TRIP_TOLERANCE + ROUND_TRIP_RTOL * |value|`,
at most 1) and the equilibrium are exact checks; the rest are what an ensemble of this
size can be asked.

In [ ]:
phi = ies.phi_measured.set_index("iteration")["mean"]
rehearsal_report = {
    "truth_round_trip": float(truth_trip["ratio"].max()),
    "start_misfit": float(start_trip["ratio"].max()),
    "write_back": float(write_back["ratio"].max()),
    "equilibrium": bool(equilibrium.converged),
    "phi_prior": float(phi.iloc[0]),
    "phi_posterior": float(phi.iloc[-1]),
    "phi_ratio": float(phi.iloc[-1] / phi.iloc[0]),
    "scalars_inside": recovered["inside"].to_dict(),
    "scalars": recovered[["truth", "posterior_p05", "posterior_p95", "z"]].to_dict("index"),
    "crisp_gap_closed": float(recovered.loc["crisp", "gap_closed"]),
    "forecast_truth": forecast_truth,
    "forecast_posterior": (float(forecast.posterior.min()), float(forecast.posterior.max())),
}
rehearsal_report["passes"] = {
    "truth_round_trip": rehearsal_report["truth_round_trip"] <= 1.0,
    "write_back": rehearsal_report["write_back"] <= 1.0,
    "equilibrium": rehearsal_report["equilibrium"],
    "phi_ratio": rehearsal_report["phi_ratio"] <= rehearsal.PHI_RATIO_MAX,
    "scalars_inside": all(rehearsal_report["scalars_inside"].values()),
    "crisp_gap_closed": rehearsal_report["crisp_gap_closed"] >= rehearsal.GAP_CLOSED_MIN,
    "forecast_covered": (rehearsal_report["forecast_posterior"][0] <= forecast_truth
                         <= rehearsal_report["forecast_posterior"][1]),
}
rehearsal_report